In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver



In [2]:
model = ChatGroq(
    model_name="openai/gpt-oss-20b",
    temperature=0
)

In [3]:
class JokeState(TypedDict):
    
    topic:str
    joke:str
    joke_explanation: str

In [5]:
def create_joke(state:JokeState):

    topic= state['topic']

    prompt=f'''
    generate a joke on the following topic: {topic}
    '''

    response= model.invoke(prompt)

    return {'joke': response.content}

In [6]:
def explain_joke(state: JokeState):

    joke= state['joke']

    prompt= f'''
    Explain the following joke to me: {joke}
    '''
    response= model.invoke(prompt)


    return {'joke_explanation': response.content}

In [7]:
graph= StateGraph(JokeState)

graph.add_node('create_joke', create_joke)
graph.add_node('explain_joke', explain_joke)


graph.add_edge(START, 'create_joke')
graph.add_edge('create_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer= InMemorySaver()




workflow= graph.compile(checkpointer= checkpointer)


In [8]:
config1= {'configurable': {'thread_id':"1"}}

response= workflow.invoke({'topic':'pizza'}, config= config1)


In [9]:
response

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄',
 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza applying for a job** | A pizza is a food, not a person, so the idea that it “applies for a job” is absurd and anthropomorphic. | The absurdity of a pizza acting like a human worker sets up the punchline. |\n| **“Crust”** | The outer edge of a pizza. | It’s a literal part of the pizza, so it’s a natural “thing” a pizza could want. |\n| **“Cost of living”** | A common phrase meaning the amount of money needed to cover basic expenses. | The joke replaces “cost” with “crust,” a word that sounds similar but is also a pizza‑specific term. |\n| **The pun** | “Crust of a living” → “Cost of living.” | The humor comes from the double meaning: the pizza wants a *crust* (money) to pay for its “living,” while also literally wanti

In [10]:
workflow.get_state(config1)


StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza applying for a job** | A pizza is a food, not a person, so the idea that it “applies for a job” is absurd and anthropomorphic. | The absurdity of a pizza acting like a human worker sets up the punchline. |\n| **“Crust”** | The outer edge of a pizza. | It’s a literal part of the pizza, so it’s a natural “thing” a pizza could want. |\n| **“Cost of living”** | A common phrase meaning the amount of money needed to cover basic expenses. | The joke replaces “cost” with “crust,” a word that sounds similar but is also a pizza‑specific term. |\n| **The pun** | “Crust of a living” → “Cost of living.” | The humor comes from the double meaning: the pizza wants a *crust* (money) to pay for its “living,” while a

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to get a *crust* of a living! 🍕😄', 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | What it is | Why it’s funny |\n|---------|------------|----------------|\n| **Pizza applying for a job** | A pizza is a food, not a person, so the idea that it “applies for a job” is absurd and anthropomorphic. | The absurdity of a pizza acting like a human worker sets up the punchline. |\n| **“Crust”** | The outer edge of a pizza. | It’s a literal part of the pizza, so it’s a natural “thing” a pizza could want. |\n| **“Cost of living”** | A common phrase meaning the amount of money needed to cover basic expenses. | The joke replaces “cost” with “crust,” a word that sounds similar but is also a pizza‑specific term. |\n| **The pun** | “Crust of a living” → “Cost of living.” | The humor comes from the double meaning: the pizza wants a *crust* (money) to pay for its “living,” while 

In [15]:
config2 = {"configurable": {"thread_id": "2"}}
response1= workflow.invoke({'topic':'pasta'}, config=config2)

In [16]:
response1

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to therapy?\n\nBecause it had too many *twists* in its life and was *knot* feeling great! 🍝😄',
 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | Literal meaning | Word‑play / pun |\n|---------|-----------------|-----------------|\n| **Spaghetti** | A long, thin pasta that is usually *twisted* or *curled* when cooked. | The word “twist” is a literal description of how spaghetti looks. |\n| **“Too many twists in its life”** | A figurative way of saying the spaghetti has had a lot of complications or dramatic turns. | “Twist” is used both literally (the shape of the pasta) and figuratively (life’s ups and downs). |\n| **“Knot”** | A knot is a tight twist or loop, like the knot you tie with a string. | It sounds exactly like “not,” so the line “was *knot* feeling great” is a homophone pun: “not” (negative feeling) + “knot” (a twist). |\n| **Therapy** | A place where people go to talk about emotional problems. | The j

In [17]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy?\n\nBecause it had too many *twists* in its life and was *knot* feeling great! 🍝😄', 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | Literal meaning | Word‑play / pun |\n|---------|-----------------|-----------------|\n| **Spaghetti** | A long, thin pasta that is usually *twisted* or *curled* when cooked. | The word “twist” is a literal description of how spaghetti looks. |\n| **“Too many twists in its life”** | A figurative way of saying the spaghetti has had a lot of complications or dramatic turns. | “Twist” is used both literally (the shape of the pasta) and figuratively (life’s ups and downs). |\n| **“Knot”** | A knot is a tight twist or loop, like the knot you tie with a string. | It sounds exactly like “not,” so the line “was *knot* feeling great” is a homophone pun: “not” (negative feeling) + “knot” (a twist). |\n| **Therapy** | A place where people go to talk about emotiona

In [18]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to therapy?\n\nBecause it had too many *twists* in its life and was *knot* feeling great! 🍝😄', 'joke_explanation': '**What’s going on in the joke?**\n\n| Element | Literal meaning | Word‑play / pun |\n|---------|-----------------|-----------------|\n| **Spaghetti** | A long, thin pasta that is usually *twisted* or *curled* when cooked. | The word “twist” is a literal description of how spaghetti looks. |\n| **“Too many twists in its life”** | A figurative way of saying the spaghetti has had a lot of complications or dramatic turns. | “Twist” is used both literally (the shape of the pasta) and figuratively (life’s ups and downs). |\n| **“Knot”** | A knot is a tight twist or loop, like the knot you tie with a string. | It sounds exactly like “not,” so the line “was *knot* feeling great” is a homophone pun: “not” (negative feeling) + “knot” (a twist). |\n| **Therapy** | A place where people go to talk about emotion